# Ear crop model: training skeleton

Trains a small CNN that takes an ear crop and predicts the 8 ear landmarks plus a visibility class (2 visible, 1 occluded, 0 absent) per landmark. Plan and file formats: `docs/ear-model-labeling.md`, `docs/ear-synthetic-data.md`.

Flow: pretrain on synthetic data, fine-tune on real labeled frames, score on real frames, export ONNX. Works on Colab or a local Jupyter (RTX 5080 needs a PyTorch build with CUDA 12.8 or later).

**Data layout** (both `SYNTH` and `REAL`): `<dir>/ear-labels.json` and `<dir>/images/<clip>/<timeMs>.jpg|png`. Labels are full-frame normalised coordinates. A frame's ear entry may carry an optional `"crop": {"cx","cy","size"}` (normalised to image height... see `crop_for`); without it the crop is derived from the labelled points, which is only a stand-in for the MediaPipe-based crop used at runtime.

In [ ]:
# --- config -------------------------------------------------------------
import json, math, random, pathlib
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageFilter
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision

CFG = dict(
    SYNTH="/content/data/synth",     # pretraining data (Blender)
    REAL="/content/data/real",       # hand-labelled real frames
    OUT="/content/out",
    SIZE=128,                         # crop size in pixels
    EPOCHS_SYNTH=10, EPOCHS_REAL=30,
    BATCH=64, LR=1e-3, SEED=0,
    WORKERS=0,                        # 0 is safe everywhere; try 2-4 on Colab/Linux (needs fork start method)
    REAL_VAL_FRACTION=0.25,           # split by clip, never by frame
    PRETRAINED=False,                 # ImageNet weights: check their licence before shipping
)
NPTS = 8
NAMES = ["helix_top","helix_back","lobe_bottom","lobe_pierce","lobe_front","tragus","antitragus","helix_root"]
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])
DEV = "cuda" if torch.cuda.is_available() else "cpu"
pathlib.Path(CFG["OUT"]).mkdir(parents=True, exist_ok=True)
print("device", DEV)

## Data

Each sample is one ear of one frame. Ears of the subject's **left** are flipped horizontally so one model handles both sides (landmark ids are anatomical, so they stay the same). Training adds random shift, scale, rotation, blur and colour jitter around the crop, to tolerate the noise of a MediaPipe-derived crop.

In [ ]:
def crop_for(w, h, pts, crop):
    """Return centre (px) and size (px) of the square crop. Prefer a given crop (normalised to image height), else derive from labelled points."""
    if crop:
        return crop["cx"] * h, crop["cy"] * h, crop["size"] * h
    xy = np.array([[p["x"] * w, p["y"] * h] for p in pts.values() if p["v"] > 0])
    lo, hi = xy.min(0), xy.max(0)
    return *((lo + hi) / 2), 1.6 * max(hi - lo)

class EarDataset(Dataset):
    def __init__(self, root, train, clips=None):
        self.root, self.train = pathlib.Path(root), train
        data = json.loads((self.root / "ear-labels.json").read_text())
        assert data["version"] == 1
        self.items = []
        for clip, e in data["clips"].items():
            if clips is not None and clip not in clips: continue
            for fr in e["frames"]:
                img = next((p for p in (self.root/"images"/clip/f"{fr['timeMs']}{x}" for x in (".jpg",".png")) if p.exists()), None)
                if img is None: continue
                for side, pts in fr["ears"].items():
                    if sum(p["v"] > 0 for p in pts.values()) >= 3:
                        self.items.append((img, side, pts, fr.get("crops", {}).get(side)))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        path, side, pts, crop = self.items[i]
        im = Image.open(path).convert("RGB"); w, h = im.size
        cx, cy, s = crop_for(w, h, pts, crop)
        th, f, fl = 0.0, 1.0, -1.0 if side == "left" else 1.0
        if self.train:
            cx += random.uniform(-.15, .15) * s; cy += random.uniform(-.15, .15) * s
            s *= random.uniform(.8, 1.3); th = math.radians(random.uniform(-15, 15))
        S, N = CFG["SIZE"], CFG["SIZE"]
        R = np.array([[math.cos(th), -math.sin(th)], [math.sin(th), math.cos(th)]]) @ np.diag([fl, 1.0])  # out->in rotation incl. flip
        c = np.array([cx, cy]); half = R @ np.array([.5, .5]) * s
        a = (s / N) * R
        data = (a[0,0], a[0,1], c[0] - half[0], a[1,0], a[1,1], c[1] - half[1])
        crop_im = im.transform((N, N), Image.AFFINE, data, resample=Image.BILINEAR)
        if self.train:
            if random.random() < .3: crop_im = crop_im.filter(ImageFilter.GaussianBlur(random.uniform(.3, 1.5)))
            x = torch.from_numpy(np.asarray(crop_im)).float() / 255
            x = (x * random.uniform(.7, 1.3) + random.uniform(-.1, .1) + torch.tensor([random.uniform(-.05, .05) for _ in range(3)])).clamp(0, 1)
        else:
            x = torch.from_numpy(np.asarray(crop_im)).float() / 255
        xy = np.zeros((NPTS, 2), np.float32); v = np.zeros(NPTS, np.int64)
        Rinv = np.linalg.inv(R)
        for k, p in pts.items():
            k = int(k); v[k] = p["v"]
            q = Rinv @ (np.array([p["x"] * w, p["y"] * h]) - c) / s + .5
            xy[k] = q
        return x.permute(2, 0, 1), torch.from_numpy(xy), torch.from_numpy(v)

def split_real(root, frac):
    clips = sorted(json.loads((pathlib.Path(root)/"ear-labels.json").read_text())["clips"])
    random.Random(0).shuffle(clips)
    k = max(1, int(len(clips) * frac)) if len(clips) > 1 else 0
    return clips[k:], clips[:k]   # train clips, val clips

## Model

MobileNetV3-small backbone, global pool, one linear head for 8 xy pairs (crop-normalised) and one for 8x3 visibility logits. Plain regression to keep the skeleton small; heatmap/SimCC heads are the likely upgrade if accuracy is short.

In [ ]:
class EarNet(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        m = torchvision.models.mobilenet_v3_small(weights="DEFAULT" if pretrained else None)
        self.features, self.pool = m.features, nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(576, 256), nn.Hardswish(), nn.Linear(256, NPTS * 5))
        self.register_buffer("mean", torch.tensor([.485, .456, .406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([.229, .224, .225]).view(1, 3, 1, 1))
    def forward(self, x):
        o = self.head(self.pool(self.features((x - self.mean) / self.std)))
        return o[:, :NPTS * 2].view(-1, NPTS, 2), o[:, NPTS * 2:].view(-1, NPTS, 3)

def loss_fn(xy_p, vis_p, xy, v):
    w = (v == 2).float() + .5 * (v == 1).float()               # absent points have no coordinate target
    l_xy = (F.smooth_l1_loss(xy_p, xy, reduction="none", beta=.02).sum(-1) * w).sum() / w.sum().clamp(min=1)
    return l_xy * 10 + F.cross_entropy(vis_p.reshape(-1, 3), v.reshape(-1))

## Train and evaluate

Evaluation reports the error in **ear heights** (helix_top to lobe_bottom distance in the crop) so it does not depend on image size, separately for visible and occluded points and for `lobe_pierce`, plus visibility accuracy.

In [ ]:
def fit(model, ds, epochs, lr):
    if epochs == 0 or len(ds) == 0: return
    dl = DataLoader(ds, CFG["BATCH"], shuffle=True, num_workers=CFG["WORKERS"], drop_last=len(ds) > CFG["BATCH"])
    opt = torch.optim.AdamW(model.parameters(), lr); sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=epochs * len(dl))
    for ep in range(epochs):
        model.train(); tot = 0
        for x, xy, v in dl:
            x, xy, v = x.to(DEV), xy.to(DEV), v.to(DEV)
            p, vp = model(x); loss = loss_fn(p, vp, xy, v)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tot += loss.item()
        print(f"epoch {ep+1}/{epochs} loss {tot/len(dl):.4f}")

@torch.no_grad()
def evaluate(model, ds):
    model.eval(); errs = {"visible": [], "occluded": [], "lobe_pierce": []}; vis_ok = vis_n = 0
    for x, xy, v in DataLoader(ds, 128):
        p, vp = model(x.to(DEV)); p = p.cpu(); pred_v = vp.argmax(-1).cpu()
        vis_ok += (pred_v == v).sum().item(); vis_n += v.numel()
        for b in range(len(x)):
            if v[b, 0] == 0 or v[b, 2] == 0: continue          # need the ear height
            eh = (xy[b, 0] - xy[b, 2]).norm().item()
            if eh < 1e-3: continue
            e = ((p[b] - xy[b]).norm(dim=-1) / eh)
            for k in range(NPTS):
                if v[b, k] == 2: errs["visible"].append(e[k].item())
                elif v[b, k] == 1: errs["occluded"].append(e[k].item())
                if k == 3 and v[b, k] > 0: errs["lobe_pierce"].append(e[k].item())
    out = {k: (float(np.mean(a)) if a else None, len(a)) for k, a in errs.items()}
    out["visibility_acc"] = vis_ok / max(vis_n, 1)
    return out

model = EarNet(CFG["PRETRAINED"]).to(DEV)
synth = EarDataset(CFG["SYNTH"], train=True) if (pathlib.Path(CFG["SYNTH"])/"ear-labels.json").exists() else []
print("synthetic samples:", len(synth))
fit(model, synth, CFG["EPOCHS_SYNTH"], CFG["LR"])

In [ ]:
real_ok = (pathlib.Path(CFG["REAL"])/"ear-labels.json").exists()
if real_ok:
    tr_clips, va_clips = split_real(CFG["REAL"], CFG["REAL_VAL_FRACTION"])
    real_tr, real_va = EarDataset(CFG["REAL"], True, tr_clips), EarDataset(CFG["REAL"], False, va_clips or tr_clips)
    print("real train/val:", len(real_tr), len(real_va), "(val falls back to train clips if there is only one clip)")
    if synth: print("synthetic-only on real val:", evaluate(model, real_va))
    fit(model, real_tr, CFG["EPOCHS_REAL"], CFG["LR"] / 3)
    print("after fine-tune on real val:", evaluate(model, real_va))

## Export

ONNX with a fixed 1x3x128x128 input in [0,1] RGB (normalisation is inside the model). Outputs: `xy` (1x8x2, crop-normalised) and `vis` (1x8x3 logits). Left ears must be flipped by the caller before inference and the x results flipped back.

In [ ]:
model.eval().cpu()
dummy = torch.zeros(1, 3, CFG["SIZE"], CFG["SIZE"])
path = pathlib.Path(CFG["OUT"]) / "ear-net.onnx"
torch.onnx.export(model, dummy, path, input_names=["image"], output_names=["xy", "vis"], opset_version=17, dynamo=False)
torch.save(model.state_dict(), pathlib.Path(CFG["OUT"]) / "ear-net.pt")
print("wrote", path, path.stat().st_size // 1024, "KB")